# D2-02 Edges fundamentals and regionalisation concepts

In this notebook, we do **not** use `edges` directly yet. Instead, we build the conceptual bridge from Day 1 by comparing a conventional characterization step with an exchange-based one.


## Learning goals

After this notebook, you should be able to:

- explain how a conventional LCIA step uses one CF per biosphere flow
- explain how an exchange-based LCIA step can use different CFs for different exchanges of the same flow
- relate that difference to locations and other edge attributes
- connect the manual matrix formulation to the motivation behind `edges`


## Background references

- Sacchi, R., Menacho, A. H., Seitfudem, G., Agez, M., Schlesinger-Martinat, J., Koyamparambath, A., Saldivar, J. S., Loubet, P., & Bauer, C. (2025). Contextual LCIA without the overhead: an exchange-based framework for flexible impact assessment. *The International Journal of Life Cycle Assessment, 30*(12), 3087-3101. https://doi.org/10.1007/s11367-025-02551-7
- Heijungs, R., & Suh, S. (2002). *The Computational Structure of Life Cycle Assessment*. Kluwer Academic Publishers. https://doi.org/10.1007/978-94-015-9900-9


## 1) Two ways to write the characterization step

In the `brightway` matrix formulation, one biosphere flow gets one characterization factor:

$$
H = QG
$$

where $Q = \operatorname{diag}(q)$ is diagonal.

In an exchange-based formulation, we proceed to a Hadamard multiplication, where characterization factors can depend on the **supplier** and the **consumer** activities at the same time:

$$
H^{\text{B}} = E^{\text{B}} \cdot G
$$

where $E^{\text{B}}$ has the same shape as $G$, and each entry can depend on edges' **consumer** or **supplier** attributes such as name, location, etc.


In [ ]:
import numpy as np
import pandas as pd

## 2) Rebuild the compact regionalized CAM system from Day 1

We reuse the same small supply chain from `D1-03` so that only the characterization logic changes.


In [ ]:
products = [
    'Li+ (aq.) [kg]',
    'Electricity (CL) [kWh]',
    'Li2CO3 [kg]',
    'Electricity (CN) [kWh]',
    'CAM [kg]',
    'Electricity (DE) [kWh]',
]
activities = [
    'Li+ extraction (CL)',
    'Hydropower (CL)',
    'Li2CO3 production (CN)',
    'Hydropower (CN)',
    'CAM production (DE)',
    'Hydropower (DE)',
]
flows = ['Water, underground [kg]', 'Water, surface [kg]', 'Water, unspecified [kg]']

A = np.array([
    [ 1.0,  0.0, -0.8, 0.0, -0.1, 0.0],
    [-0.5,  1.0,  0.0, 0.0,-25.0, 0.0],
    [ 0.0,  0.0,  1.0, 0.0, -0.2, 0.0],
    [ 0.0,  0.0, -3.0, 1.0,  0.0, 0.0],
    [ 0.0,  0.0,  0.0, 0.0,  1.0, 0.0],
    [ 0.0,  0.0,  0.0, 0.0,-25.0, 1.0],
])

B = np.array([
    [2.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 3.0, 0.0, 0.0, 0.0],
    [0.0, 0.02, 0.0, 0.015, 0.2, 0.005],
])

f = np.array([0.0, 0.0, 0.0, 0.0, 1.0, 0.0])
q_global = np.array([39.5, 39.5, 39.5])

display(pd.DataFrame(A, index=products, columns=activities))
display(pd.DataFrame(B, index=flows, columns=activities))
display(pd.DataFrame(f, index=products, columns=['Demand']))
display(pd.DataFrame(q_global, index=flows, columns=['Global CF']))


## 3) Conventional characterization with one global factor per flow

This reproduces the Day 1 logic: the same water-withdrawal factor is applied everywhere.


In [ ]:
s = np.linalg.solve(A, f)
G = B @ np.diag(s)
Q_global = np.diag(q_global)
H_global = Q_global @ G

print('Scaling vector s')
display(pd.DataFrame(s, index=activities, columns=['Scale']))

print('Inventory matrix G')
display(pd.DataFrame(G, index=flows, columns=activities))

print('Characterized inventory with global CFs')
display(pd.DataFrame(H_global, index=flows, columns=activities))

print(f'Total score with global CFs: {H_global.sum():.1f}') # let's display only one decimal


## 4) Exchange-based characterization with location-specific AWARE factors

We now mirror the supplementary CAM example more closely.

The global case used one factor everywhere: `39.5`.

In the exchange-based case, the factor depends on the location of the activity where the water withdrawal occurs (the **consumer**), according to `AWARE 2.0`:

- Chile (`CL`): `45.5`
- China (`CN`): `6.3`
- Germany (`DE`): `2.1`

So $E^B$ has the same shape as `G`, but its non-zero entries now depend on the activity location attached to each exchange.


In [ ]:
G_df = pd.DataFrame(G, index=flows, columns=activities)

activity_locations = pd.Series(
    {
        'Li+ extraction (CL)': 'CL',
        'Hydropower (CL)': 'CL',
        'Li2CO3 production (CN)': 'CN',
        'Hydropower (CN)': 'CN',
        'CAM production (DE)': 'DE',
        'Hydropower (DE)': 'DE',
    },
    name='Location',
)
aware_by_location = {'CL': 45.5, 'CN': 6.3, 'DE': 2.1}

location_factors = pd.DataFrame(
    {
        'Location': ['CL', 'CN', 'DE'],
        'AWARE 2.0 CF': [45.5, 6.3, 2.1],
    }
)

display(location_factors)
display(activity_locations.to_frame())

E_B = pd.DataFrame(0.0, index=flows, columns=activities)

for activity, location in activity_locations.items():
    E_B.loc[G_df[activity] != 0.0, activity] = aware_by_location[location]

H_edges = E_B * G_df

print('Exchange-based characterization matrix E^B')
display(E_B)

print('Characterized inventory with exchange-based CFs')
display(H_edges)

print(f'Total score with exchange-based CFs: {H_edges.sum().sum():.2f}.')


Do you notice the difference in score? If so, why is that?

## Checkpoint 1

Compare the two totals and answer:

- Which activity columns change between the conventional and exchange-based results?
- Why can the same biosphere flow, such as `Water, unspecified [kg]`, receive different factors in different columns?


In [ ]:
# TODO
# global_total = ...
# edges_total = ...
# changed_columns = ...


In [ ]:
global_total = float(H_global.sum())
changed = H_edges - pd.DataFrame(H_global, index=flows, columns=activities)
changed_columns = changed.sum(axis=0)
changed_columns = changed_columns[changed_columns != 0].sort_values(ascending=False)

print('Global total:', global_total)
print('Exchange-based total:', float(H_edges.to_numpy().sum()))
print('Changed columns:')
display(changed_columns.to_frame('Delta score'))
print('Explanation: the factor is attached to each flow-activity exchange, so the same flow can receive different CFs depending on where the consuming activity is located.')


## 5) Put the two formulations side by side

The practical difference is easiest to see in the shapes of the objects we apply to the inventory.


In [ ]:
comparison = pd.DataFrame([
    {
        'object': 'Global CF vector q',
        'shape': str(q_global.shape),
        'meaning': 'One CF per biosphere flow',
    },
    {
        'object': 'Diagonal matrix Q = diag(q)',
        'shape': str(Q_global.shape),
        'meaning': 'Used in conventional LCIA',
    },
    {
        'object': 'Inventory matrix G',
        'shape': str(G_df.shape),
        'meaning': 'One inventory entry per flow-activity pair',
    },
    {
        'object': 'Exchange-based CF matrix E^B',
        'shape': str(E_B.shape),
        'meaning': 'One CF per eligible exchange, driven here by activity location',
    },
])
comparison


## 6) Why location lives on the edge

In this CAM example, the biosphere flows are still the same three water-withdrawal flows, but their characterization differs because the withdrawals happen in different activity locations.

That is why `edges` uses edge attributes from both ends of the exchange. The characterization factor can depend on:

- the supplier flow identity
- the consumer activity identity
- the geography attached to the consumer or supplier
- other activity metadata used in the method definition


In [ ]:
edge_attributes = (
    G_df.stack()
    .rename('Inventory amount')
    .reset_index()
    .rename(columns={'level_0': 'flow', 'level_1': 'consumer activity'})
)
edge_attributes = edge_attributes[edge_attributes['Inventory amount'] != 0].copy()
edge_attributes['consumer location'] = edge_attributes['consumer activity'].map(activity_locations)
edge_attributes['CF used'] = [E_B.loc[row['flow'], row['consumer activity']] for _, row in edge_attributes.iterrows()]
edge_attributes


## Recap

After this notebook, you should now be able to:

- explain why a diagonal characterization matrix is limited for regionalized LCIA
- construct a simple exchange-based CF matrix by hand from location-specific factors
- compare `Q @ G` with an element-wise `E^B * G` formulation
- explain why location is an edge attribute in this context
